In [1]:
import pandas as pd
import os, json, librosa
import soundfile as sf
import io

In [ ]:
df = pd.read_parquet(r'/home/nextai/asr-nepali/Nepali-ASR-FineTuning/rishi70612_audio_data.parquet')
print(df.columns)
print(df.iloc[0])

input_ids→ tokens created for another model,
labels→ likely TTS training data,
input_length→ related to previous processing,
speaker_embedding→ information about the speaker's voice,
path→ old Windows path, probably unusable in Colab

In [ ]:
for i in range(10):
    audio_bytes = df.iloc[i]['audio']['bytes']
    data, sr = sf.read(io.BytesIO(audio_bytes))
    print(i, sr) # Checking whether the audio frequency is 16khz or not

In [ ]:
# Only these two columns matter
df = df[['audio', 'sentence']]

In [ ]:
audio_dir = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/tuning_audio'
os.makedirs(audio_dir, exist_ok=True)
manifest_path = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/manifest.json'

with open(manifest_path, 'w', encoding='utf-8') as f:
    for i, row in df.iterrows():
        audio_bytes = row['audio']['bytes']
        wav_path = f"{audio_dir}/{i}.wav"
        data, sr = sf.read(io.BytesIO(audio_bytes))

        if sr != 16000:
          data = librosa.resample(data, orig_sr=sr, target_sr=16000)
          sr = 16000

        sf.write(wav_path, data, sr)
        duration = len(data) / sr

        entry = {
            "audio_filepath": wav_path,
            "text": row['sentence'],
            "duration": duration,
            "target_lang": "ne-NP"
        }
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

In [ ]:
with open(manifest_path) as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

In [ ]:
import random

with open(manifest_path, encoding='utf-8') as f:
    lines = f.readlines()

random.seed(42)
random.shuffle(lines)

split_idx = int(len(lines) * 0.9)  # 90% train, 10% val
train_lines = lines[:split_idx]
val_lines = lines[split_idx:]

print(f"Total: {len(lines)}, Train: {len(train_lines)}, Val: {len(val_lines)}")

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json', 'w', encoding='utf-8') as f:
    f.writelines(train_lines)

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json', 'w', encoding='utf-8') as f:
    f.writelines(val_lines)

In [ ]:
with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

In [2]:
import nemo
import nemo.collections.asr as nemo_asr
print(nemo.__version__)

d:\FineTuning\.myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.


3.1.0+8e683a052


In [1]:
!git clone -b main https://github.com/NVIDIA/NeMo.git "d:/FineTuning/NeMo"

Cloning into 'd:/FineTuning/NeMo'...
Updating files:  67% (1570/2316)
Updating files:  68% (1575/2316)
Updating files:  69% (1599/2316)
Updating files:  70% (1622/2316)
Updating files:  71% (1645/2316)
Updating files:  72% (1668/2316)
Updating files:  73% (1691/2316)
Updating files:  74% (1714/2316)
Updating files:  75% (1737/2316)
Updating files:  76% (1761/2316)
Updating files:  77% (1784/2316)
Updating files:  78% (1807/2316)
Updating files:  79% (1830/2316)
Updating files:  80% (1853/2316)
Updating files:  81% (1876/2316)
Updating files:  82% (1900/2316)
Updating files:  83% (1923/2316)
Updating files:  84% (1946/2316)
Updating files:  85% (1969/2316)
Updating files:  86% (1992/2316)
Updating files:  87% (2015/2316)
Updating files:  88% (2039/2316)
Updating files:  89% (2062/2316)
Updating files:  90% (2085/2316)
Updating files:  91% (2108/2316)
Updating files:  92% (2131/2316)
Updating files:  93% (2154/2316)
Updating files:  94% (2178/2316)
Updating files:  95% (2201/2316)
Updati

In [ ]:
# !ls "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/NeMo/examples/asr"

In [ ]:
from nemo.collections.asr.models import ASRModel

model_path = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo"

model = ASRModel.restore_from(model_path)


In [ ]:
!tar -tf "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo"

In [ ]:
!mkdir -p /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer
!tar -xf "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" \
  -C /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer "427ad33c6285472cb01c3eb843d2309d_tokenizer.model"
!mv "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/427ad33c6285472cb01c3eb843d2309d_tokenizer.model" \
    "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/tokenizer.model"
!ls -la /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer

!tar -xf "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" \
  -C /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer "c8ead90c911846569df4738620acfc0f_vocab.txt"
!mv "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/c8ead90c911846569df4738620acfc0f_vocab.txt" \
    "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/vocab.txt"
!ls -la /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer


In [ ]:
print(model_cfg['tokenizer'])

In [ ]:
!tar -xf "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo" -C /home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer "model_config.yaml"

In [ ]:
import yaml

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)
model_cfg['tokenizer'] = {
    'dir': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer',
    'type': 'bpe'
}

model_cfg['train_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json' # Point data at manifests
model_cfg['train_ds']['is_tarred'] = False   # using a plain manifest, not tarred shards
model_cfg['train_ds']['shuffle']=True
model_cfg['train_ds']['batch_duration'] = 20
model_cfg['validation_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json'
model_cfg['test_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json' # It exists because the model configuration expects a test dataset configuration, even if you don't use it during training.
model_cfg["optim"]["lr"] = 0.0001
model_cfg['validation_ds']['default_prompt_mode'] = 'unified'
model_cfg['validation_ds']['unified_auto_ratio'] = 0.0
model_cfg['validation_ds']['lang_field'] = 'target_lang'
model_cfg['test_ds']['default_prompt_mode'] = 'unified'
model_cfg['test_ds']['unified_auto_ratio'] = 0.0
model_cfg['test_ds']['lang_field'] = 'target_lang'
model_cfg['decoding']['greedy']['use_cuda_graph_decoder'] = False # Hello
full_cfg = {
    'name': 'FastConformer-Transducer-BPE-Prompt-Streaming-Pilot',
    'model': model_cfg,
    'trainer': {
        'devices': 1, 'num_nodes': 1, 'max_epochs': -1,   # moot with Lhotse; -1 avoids implying it matters
        'accelerator': 'auto', 'precision': 'fp32',
        'log_every_n_steps': 10, 'enable_progress_bar': True,
        'num_sanity_val_steps': 0, 'enable_checkpointing': False,
        'logger': False, 'use_distributed_sampler': False,
        'max_steps': 250,
        'limit_train_batches': 50,   # ~5 pseudo-epochs over 250 steps -> 5 validation/checkpoint points
},
    'exp_manager': {
        'exp_dir': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run',
        'create_tensorboard_logger': True,
        'create_checkpoint_callback': True,
        'checkpoint_callback_params': {'monitor': 'val_wer', 'mode': 'min', 'save_top_k': 1, 'always_save_nemo': True},
        'resume_if_exists': False,
        'resume_ignore_no_checkpoint':True,
    },
    'init_from_nemo_model': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo',
}

out_path = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/finetune_config.yaml'
with open(out_path, 'w', encoding='utf-8') as f:
    yaml.dump(full_cfg, f, allow_unicode=True, sort_keys=False)

print("Saved:", out_path)

In [ ]:
import yaml

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer/model_config.yaml') as f:
    model_cfg = yaml.safe_load(f)

model_cfg['tokenizer'] = {
    'dir': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/extracted_tokenizer',
    'type': 'bpe'
}

# Data
model_cfg['train_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json'
model_cfg['train_ds']['is_tarred'] = False
model_cfg['train_ds']['shuffle'] = True
model_cfg['train_ds']['batch_duration'] = 20
model_cfg['validation_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json'
model_cfg['test_ds']['manifest_filepath'] = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/val_manifest.json'

# Prompt/language routing
model_cfg['validation_ds']['default_prompt_mode'] = 'unified'
model_cfg['validation_ds']['unified_auto_ratio'] = 0.0
model_cfg['validation_ds']['lang_field'] = 'target_lang'
model_cfg['test_ds']['default_prompt_mode'] = 'unified'
model_cfg['test_ds']['unified_auto_ratio'] = 0.0
model_cfg['test_ds']['lang_field'] = 'target_lang'

model_cfg['decoding']['greedy']['use_cuda_graph_decoder'] = False

# --- Optimizer: replaces flat lr=0.0001 with warmup + a real peak LR ---
# NoamAnnealing needs d_model to scale correctly — pull it from the encoder
# config rather than guessing, since a wrong value defeats the point of Noam.
d_model = model_cfg.get('encoder', {}).get('d_model')

if d_model:
    model_cfg['optim'] = {
        'name': 'adamw',
        'lr': 0.005,              # peak LR, only reached after warmup ramp
        'betas': [0.9, 0.98],
        'weight_decay': 1e-3,
        'sched': {
            'name': 'NoamAnnealing',
            'd_model': d_model,
            'warmup_steps': 200,
            'min_lr': 1e-6,
        },
    }
else:
    # Fallback: CosineAnnealing doesn't need d_model, safer if the encoder
    # config doesn't expose it under this key.
    model_cfg['optim'] = {
        'name': 'adamw',
        'lr': 0.005,
        'betas': [0.9, 0.98],
        'weight_decay': 1e-3,
        'sched': {
            'name': 'CosineAnnealing',
            'warmup_steps': 200,
            'min_lr': 1e-6,
        },
    }

full_cfg = {
    'name': 'FastConformer-Transducer-BPE-Prompt-Streaming-Pilot',
    'model': model_cfg,
    'trainer': {
        'devices': 1, 'num_nodes': 1, 'max_epochs': -1,
        'accelerator': 'auto', 'precision': 'fp32',
        'log_every_n_steps': 10, 'enable_progress_bar': True,
        'num_sanity_val_steps': 0, 'enable_checkpointing': False,
        'logger': False, 'use_distributed_sampler': False,
        'max_steps': 250,
        'limit_train_batches': 50,       # kept as int — 1.0 breaks on Lhotse's infinite dataloader
        'gradient_clip_val': 1.0,        # caps any single spike, given the step-49 blowup we found
    },
    'exp_manager': {
        'exp_dir': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run',
        'create_tensorboard_logger': True,
        'create_checkpoint_callback': True,
        'checkpoint_callback_params': {'monitor': 'val_wer', 'mode': 'min', 'save_top_k': 1, 'always_save_nemo': True},
        'resume_if_exists': False,
        'resume_ignore_no_checkpoint': True,
    },
    'init_from_nemo_model': '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo',
}

out_path = '/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/finetune_config.yaml'
with open(out_path, 'w', encoding='utf-8') as f:
    yaml.dump(full_cfg, f, allow_unicode=True, sort_keys=False)

print("Saved:", out_path)
print("Using scheduler:", model_cfg['optim']['sched']['name'], "| d_model:", d_model)

!python "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/NeMo/examples/asr/asr_transducer/speech_to_text_rnnt_bpe_prompt.py" \
  --config-path="/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run" \
  --config-name="finetune_config"

In [ ]:
from tensorboard.backend.event_processing import event_accumulator

ea = event_accumulator.EventAccumulator('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/default/<new_run_timestamp>')
ea.Reload()
print(ea.Tags()['scalars'])

loss_events = ea.Scalars('train_loss')
for e in loss_events:
    print(e.step, e.value)

In [ ]:
import tarfile
import torch
import io

nemo_path = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/nemotron-3.5-asr-streaming-0.6b.nemo"

with tarfile.open(nemo_path, "r") as tar:
    f = tar.extractfile("model_weights.ckpt")
    ckpt = torch.load(io.BytesIO(f.read()), map_location="cpu")

print(type(ckpt))
print(ckpt.keys())

In [ ]:
state_dict = ckpt["state_dict"] if "state_dict" in ckpt else ckpt

found = False
for key in state_dict.keys():
    if "prompt_kernel" in key:
        print(key)
        found = True

print("\nFound:", found)

In [ ]:
import torch

nan_params = []

for name, p in model.named_parameters():
    if torch.isnan(p).any():
        nan_params.append(name)

print("Number of parameters containing NaNs:", len(nan_params))

if nan_params:
    print("\nFirst few parameters with NaNs:")
    for name in nan_params[:20]:
        print(name)

In [ ]:
import torch

ckpt = torch.load("model_weights.ckpt", map_location="cpu")

for key in ckpt["state_dict"]:
    if "prompt_kernel" in key:
        print(key)

!python "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/NeMo/examples/asr/asr_transducer/speech_to_text_rnnt_bpe_prompt.py" \
  --config-path="/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run" \
  --config-name="finetune_config" 2>&1 | tee /home/nextai/asr-nepali/Nepali-ASR-FineTuning/full_run_capture.txt

In [ ]:
!wc -l /home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json

In [ ]:
import json

dur = 0
count = 0
with open("/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json") as f:
    for line in f:
        obj = json.loads(line)
        dur += obj["duration"]
        count += 1

print("Files:", count)
print("Total hours:", dur/3600)
print("Average duration:", dur/count)

In [ ]:
from tensorboard.backend.event_processing import event_accumulator

ea = event_accumulator.EventAccumulator('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/default/2026-07-15_08-36-30')
ea.Reload()

print(ea.Tags()['scalars'])  # see what metrics were logged (train_loss, val_loss, val_wer, etc.)

In [ ]:
loss_events = ea.Scalars('train_loss')
for e in loss_events:
    print(e.step, e.value)

In [ ]:
import json

with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/tuning_run/train_manifest.json', encoding='utf-8') as f:
    lines = f.readlines()

bad = []
for i, line in enumerate(lines):
    entry = json.loads(line)
    val = entry.get('target_lang')
    if not val or not isinstance(val, str) or not val.strip():
        bad.append((i, entry.get('audio_filepath'), repr(val)))

print('Total lines:', len(lines))
print('Bad entries:', len(bad))
for b in bad[:10]:
    print(b)

In [ ]:
import soundfile as sf

bad_audio = []
for line in lines:
    entry = json.loads(line)
    path = entry['audio_filepath']
    try:
        info = sf.info(path)
        if info.frames == 0:
            bad_audio.append(path)
    except Exception as e:
        bad_audio.append((path, str(e)))

print('Bad/unreadable audio files:', len(bad_audio))
print(bad_audio[:10])

In [ ]:
!awk '/^  train_ds:/,/^  validation_ds:/' ~/asr-nepali/Nepali-ASR-FineTuning/finetune_config.yaml | grep -i "lang_field\|prompt_field"

In [ ]:
import json
paths = [json.loads(l)['audio_filepath'] for l in lines]
print('Duplicates:', len(paths) - len(set(paths)))

## Testing (Not tested)

In [ ]:
manifest_path = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/pilot_manifest.json"
test_manifest_path = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/test_manifest.json"

with open(manifest_path, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)

        if entry["audio_filepath"].endswith("/1025.wav"):
            with open(test_manifest_path, 'w', encoding='utf-8') as out:
                out.write(json.dumps(entry, ensure_ascii=False) + '\n')

            print(entry)
            break

In [ ]:
with open('/home/nextai/asr-nepali/Nepali-ASR-FineTuning/test_manifest.json') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 3: break

In [ ]:
from nemo.collections.asr.models.rnnt_bpe_models_prompt import EncDecRNNTBPEModelWithPrompt

finetuned_model = EncDecRNNTBPEModelWithPrompt.restore_from(
    "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/pilot_run/default/checkpoints/default.nemo"
)
finetuned_model = finetuned_model.cuda().eval()

In [ ]:
import lightning.pytorch as pl
from omegaconf import OmegaConf

finetuned_model.cfg.test_ds.manifest_filepath = "/home/nextai/asr-nepali/Nepali-ASR-FineTuning/test_manifest.json"
finetuned_model.setup_test_data(finetuned_model.cfg.test_ds)

trainer = pl.Trainer(accelerator="gpu", devices=1, logger=False)
trainer.test(finetuned_model)